# Segment Tree & BIT (Fenwick Tree) — Technical Reference

## Quick Index

| Technique | When to use | Section |
| :--- | :--- | :--- |
| BIT (Fenwick Tree) | Point update + prefix sum query | §1 |
| BIT — range update | Range increment + point query | §2 |
| Segment Tree | Range query + point or range update | §3 |
| Segment Tree — lazy propagation | Range update + range query efficiently | §4 |

---
## When to Use

| Signal | Structure to reach for |
| :--- | :--- |
| "range sum query" with point updates | BIT |
| "count elements less than x" dynamically | BIT (coordinate compression) |
| "range minimum/maximum query" | Segment Tree |
| "range update + range query" | Segment Tree with lazy propagation |
| "how many inversions" | BIT |
| Static array, range queries only, no updates | Prefix sum (simpler — no BIT/SegTree needed) |

---
## BIT vs Segment Tree — When to Choose

| | BIT (Fenwick Tree) | Segment Tree |
| :--- | :--- | :--- |
| **Implementation complexity** | Simple — 10 lines | Moderate — 30–50 lines |
| **Supported operations** | Prefix sum, point update | Any associative operation (sum, min, max, gcd) |
| **Range update** | Possible with difference BIT trick | Native with lazy propagation |
| **Memory** | O(n) | O(4n) |
| **Speed** | Faster in practice | Slightly slower but more flexible |
| **When to use** | Sum queries + point updates | Min/max queries or range updates needed |

---
## §1 — BIT (Fenwick Tree) — Point Update + Prefix Sum Query

A BIT stores partial sums in a 1-indexed array. Each index `i` stores the sum of a range determined by the lowest set bit of `i`. Point update and prefix query both run in O(log n).

**The two core operations:**
- `update(i, delta)` — add `delta` to index `i` and propagate up using `i += i & (-i)`
- `query(i)` — return prefix sum `[1..i]` by accumulating down using `i -= i & (-i)`

`i & (-i)` isolates the lowest set bit of `i` — this is the entire magic of the BIT.

In [ ]:
# BIT — 1-indexed, size n
class BIT:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)      # 1-indexed

    def update(self, i, delta):
        while i <= self.n:
            self.tree[i] += delta
            i += i & (-i)              # move to parent

    def query(self, i):                # prefix sum [1..i]
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & (-i)              # move to predecessor
        return total

    def range_query(self, l, r):       # sum [l..r]
        return self.query(r) - self.query(l - 1)


# Build BIT from existing array — O(n log n)
def build(nums):
    bit = BIT(len(nums))
    for i, val in enumerate(nums):
        bit.update(i + 1, val)         # 1-indexed
    return bit


# LC 307 — Range Sum Query Mutable
# Point update + range sum query
class NumArray:
    def __init__(self, nums):
        self.nums = nums[:]
        self.bit = BIT(len(nums))
        for i, v in enumerate(nums):
            self.bit.update(i + 1, v)

    def update(self, index, val):
        self.bit.update(index + 1, val - self.nums[index])  # update delta only
        self.nums[index] = val

    def sumRange(self, left, right):
        return self.bit.range_query(left + 1, right + 1)

**Common mistakes:**
- Using 0-indexed BIT — BIT must be 1-indexed; `i & (-i)` returns 0 when `i = 0`, causing infinite loop
- Passing absolute value instead of delta to `update` — always pass the difference `(new_val - old_val)`
- Confusing `update` direction (`i += i & (-i)`) with `query` direction (`i -= i & (-i)`) — they are opposite

Problems: LC 307, LC 315, LC 493

---
## §2 — BIT — Range Update + Point Query

By storing a difference array in the BIT, you can add `delta` to a range `[l, r]` in O(log n) and query a single point in O(log n). This is the inverse of the standard BIT.

In [ ]:
# BIT difference array — range update + point query
class BIT_RangeUpdate:
    def __init__(self, n):
        self.tree = [0] * (n + 2)
        self.n = n

    def _update(self, i, delta):
        while i <= self.n:
            self.tree[i] += delta
            i += i & (-i)

    def _query(self, i):
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & (-i)
        return total

    def range_update(self, l, r, delta):   # add delta to [l..r]
        self._update(l, delta)
        self._update(r + 1, -delta)        # cancel after r

    def point_query(self, i):              # value at point i
        return self._query(i)

**Common mistakes:**
- Forgetting `_update(r + 1, -delta)` — without it, the update extends past `r`
- Using this for range query — this variant only supports point queries; use standard BIT or Segment Tree for range queries after range updates

Problems: LC 1109, LC 370

---
## §3 — Segment Tree — Range Query + Point Update

A Segment Tree is a binary tree where each node stores the aggregate (sum, min, max) of a range. Leaves store individual elements. Build in O(n), update and query in O(log n).

**Node indexing:** for node at index `i`, left child is `2*i`, right child is `2*i+1`. Tree size is `4*n` to be safe.

In [ ]:
# Segment Tree — sum queries, point updates
class SegmentTree:
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)
        self._build(nums, 0, 0, self.n - 1)

    def _build(self, nums, node, start, end):
        if start == end:
            self.tree[node] = nums[start]
        else:
            mid = (start + end) // 2
            self._build(nums, 2*node+1, start, mid)      # left child
            self._build(nums, 2*node+2, mid+1, end)      # right child
            self.tree[node] = self.tree[2*node+1] + self.tree[2*node+2]  # merge

    def update(self, idx, val, node=0, start=0, end=None):
        if end is None: end = self.n - 1
        if start == end:
            self.tree[node] = val
        else:
            mid = (start + end) // 2
            if idx <= mid:
                self.update(idx, val, 2*node+1, start, mid)
            else:
                self.update(idx, val, 2*node+2, mid+1, end)
            self.tree[node] = self.tree[2*node+1] + self.tree[2*node+2]

    def query(self, l, r, node=0, start=0, end=None):    # range sum [l..r]
        if end is None: end = self.n - 1
        if r < start or end < l: return 0                # out of range
        if l <= start and end <= r: return self.tree[node]  # fully inside
        mid = (start + end) // 2
        return (self.query(l, r, 2*node+1, start, mid) +
                self.query(l, r, 2*node+2, mid+1, end))

**To change from sum to min/max:** replace the merge line `self.tree[node] = left + right` with `min(left, right)` or `max(left, right)`, and return `float('inf')` (for min) or `float('-inf')` (for max) when out of range.

**Common mistakes:**
- Allocating `2*n` nodes instead of `4*n` — causes index out of bounds for non-power-of-2 sizes
- Returning wrong sentinel for out-of-range case — return `0` for sum, `inf` for min, `-inf` for max
- Forgetting to update the parent node after updating a child — always recompute `tree[node]` after recursing

Problems: LC 307, LC 315, LC 732

---
## §4 — Segment Tree with Lazy Propagation — Range Update + Range Query

Lazy propagation defers range updates — instead of immediately updating all nodes in a range, store a pending update (`lazy`) at the node and push it down to children only when needed. This enables range updates in O(log n).

In [ ]:
# Segment Tree with lazy propagation — range update + range sum query
class LazySegmentTree:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (4 * n)
        self.lazy = [0] * (4 * n)     # pending updates

    def _push_down(self, node, start, end):
        if self.lazy[node] != 0:
            mid = (start + end) // 2
            # apply pending update to children
            self.tree[2*node+1] += self.lazy[node] * (mid - start + 1)
            self.tree[2*node+2] += self.lazy[node] * (end - mid)
            self.lazy[2*node+1] += self.lazy[node]
            self.lazy[2*node+2] += self.lazy[node]
            self.lazy[node] = 0       # clear pending update

    def update(self, l, r, delta, node=0, start=0, end=None):  # add delta to [l..r]
        if end is None: end = self.n - 1
        if r < start or end < l: return
        if l <= start and end <= r:   # fully inside — apply lazily
            self.tree[node] += delta * (end - start + 1)
            self.lazy[node] += delta
            return
        self._push_down(node, start, end)
        mid = (start + end) // 2
        self.update(l, r, delta, 2*node+1, start, mid)
        self.update(l, r, delta, 2*node+2, mid+1, end)
        self.tree[node] = self.tree[2*node+1] + self.tree[2*node+2]

    def query(self, l, r, node=0, start=0, end=None):
        if end is None: end = self.n - 1
        if r < start or end < l: return 0
        if l <= start and end <= r: return self.tree[node]
        self._push_down(node, start, end)  # push before querying children
        mid = (start + end) // 2
        return (self.query(l, r, 2*node+1, start, mid) +
                self.query(l, r, 2*node+2, mid+1, end))

**Common mistakes:**
- Forgetting to call `_push_down` before recursing into children during both update and query — stale lazy values produce wrong results
- Not multiplying `lazy[node]` by range size when updating `tree[node]` — the tree stores sums, not per-element values
- Forgetting to clear `lazy[node]` after pushing down — causes double application of the pending update

Problems: LC 307, LC 732, LC 1622